## CNN Training

### import datasets

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import torchvision.datasets as datasets

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

tr_set = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
tr_loader = data.DataLoader(tr_set, batch_size=64, shuffle=True)
ts_set = datasets.MNIST(root='./data', train=False, transform=transform, download=True)
ts_loader = data.DataLoader(ts_set, batch_size=64, shuffle=True)

### define CNN model

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # define CNN ( The input channel is 1 and the output channel is 32. Kernel size: 3x3)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        # define CNN (The input channel is 32, Output channel is 64)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # 定义全连接层
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 64 * 7 * 7)  # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

cnn_model = SimpleCNN()

### define optimizer and loss function

In [17]:
criteron = nn.CrossEntropyLoss()        # 多分类交叉熵损失
optimizer = optim.SGD(cnn_model.parameters(), lr=0.001, momentum=0.9)

### train

In [19]:
num_epochs = 5
cnn_model.train()

for epoch in range(num_epochs):
    total_loss = 0
    for imgs, lbs in tr_loader:
        outputs = cnn_model(imgs)       # forward
        loss = criteron(outputs, lbs)   # cal loss

        optimizer.zero_grad()
        loss.backward()     # backward
        optimizer.step()

        total_loss += loss.item()
    print(f'Epoch {epoch+1}: Loss: {total_loss/len(tr_loader):.3f}')

KeyboardInterrupt: 

### evaluate model

In [20]:
cnn_model.eval()
crct = 0
ttl = 0

with torch.no_grad():
    for imgs, lbs in ts_loader:
        outputs = cnn_model(imgs)
        _, pre = torch.max(outputs, 1)
        ttl += lbs.size(0)
        crct += (pre == lbs).sum().item()

accuracy = 100 * crct / ttl
print(f'Accuracy: {accuracy:.2f}%')

Accuracy: 94.28%
